[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laverde97/phd-data-science-ai/blob/main/semesters/semester-01/machine-learning/notes/notebooks/06-arbol-vs-random-forest-clasificacion.ipynb)


# Comparativo: Árbol de Decisión vs Random Forest para Clasificación

## ¿Cuál de los dos modelos clasifica mejor?

### Objetivo

Comparar de forma **justa, reproducible y didáctica**:

1. **DecisionTreeClassifier**
2. **RandomForestClassifier**

Usaremos:

- el mismo dataset;
- la misma división Train/Test;
- `stratify=y`;
- la misma validación cruzada;
- las mismas métricas;
- optimización independiente de hiperparámetros;
- las mismas observaciones de Test.

### Métrica principal

Usaremos **F1 macro** porque:

- considera ambas clases;
- evita depender únicamente de cuál clase fue codificada como positiva;
- equilibra Precision y Recall.

También analizaremos:

- Accuracy
- Precision macro
- Recall macro
- F1 macro
- ROC-AUC
- Recall de la clase maligna
- matrices de confusión
- curvas ROC
- curvas Precision-Recall
- predicciones individuales
- errores por observación
- Train vs Test
- validación cruzada
- tiempo de búsqueda

> El mejor modelo será el mejor **para este dataset y bajo este protocolo**, no necesariamente para todos los problemas de clasificación.



# 1. Diferencia conceptual

## Árbol de Decisión

Utiliza un único árbol de reglas.

### Ventajas
- alta interpretabilidad;
- rápido;
- fácil de visualizar.

### Desventajas
- puede ser inestable;
- puede sobreajustarse;
- alta varianza.

## Random Forest

Combina muchos árboles.

### Ventajas
- mayor estabilidad;
- menor varianza;
- suele generalizar mejor;
- permite OOB Score.

### Desventajas
- menor interpretabilidad;
- mayor costo computacional.



# 2. Dataset real: Breast Cancer Wisconsin

Usaremos exactamente los mismos datos de los dos Colabs anteriores.

- **569 observaciones**
- **30 variables predictoras**
- dos clases:
  - `0 = malignant`
  - `1 = benign`

Esto es importante porque, por defecto, muchas métricas binarias de scikit-learn consideran como positiva la clase `1`.

Por eso en este comparativo usaremos:

- métricas **macro** para comparar ambas clases de forma simétrica;
- y además calcularemos explícitamente el **Recall de la clase maligna**.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    classification_report
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:

data = load_breast_cancer(as_frame=True)

X = data.data.copy()
y = data.target.copy()

df = X.copy()
df["target"] = y

print("Dimensiones:", df.shape)
print("Clases:", dict(enumerate(data.target_names)))
display(df.head())


# 3. Exploración rápida

In [ ]:

print("Valores faltantes totales:", df.isna().sum().sum())

conteo = y.value_counts().sort_index()

tabla_clases = pd.DataFrame({
    "Clase": [data.target_names[i] for i in conteo.index],
    "Cantidad": conteo.values,
    "Porcentaje": conteo.values / len(y) * 100
})

display(tabla_clases.round(2))


In [ ]:

plt.figure(figsize=(7,5))
plt.bar(tabla_clases["Clase"], tabla_clases["Cantidad"])
plt.title("Distribución de clases")
plt.xlabel("Clase")
plt.ylabel("Cantidad")
plt.show()



# 4. Train/Test estratificado

Ambos modelos deben recibir exactamente las mismas observaciones.

Usaremos:

- 80% Train
- 20% Test
- `stratify=y`


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nDistribución Train:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nDistribución Test:")
print(y_test.value_counts(normalize=True).sort_index())



# 5. Métricas comunes

## Accuracy
Porcentaje total de predicciones correctas.

## Precision macro
Calcula Precision para cada clase y luego promedia.

## Recall macro
Calcula Recall para cada clase y luego promedia.

## F1 macro
Calcula F1 para cada clase y luego promedia.

## ROC-AUC
Mide discriminación global usando probabilidades.

## Recall maligno
Mide qué proporción de casos malignos reales detecta correctamente el modelo.

En un problema de detección, esta última métrica puede ser especialmente importante.


In [ ]:

def metricas_clasificacion(y_true, y_pred, y_proba_clase1):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(
            y_true, y_pred, average="macro"
        ),
        "Recall_macro": recall_score(
            y_true, y_pred, average="macro"
        ),
        "F1_macro": f1_score(
            y_true, y_pred, average="macro"
        ),
        "ROC_AUC": roc_auc_score(
            y_true, y_proba_clase1
        ),
        "Recall_malignant": recall_score(
            y_true, y_pred, pos_label=0
        )
    }



# 6. Validación cruzada común

Usaremos `StratifiedKFold` para que cada fold conserve aproximadamente la proporción de clases.


In [ ]:

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# 7. Optimización del Árbol de Decisión

In [ ]:

arbol = DecisionTreeClassifier(
    random_state=RANDOM_STATE
)

param_grid_arbol = {
    "criterion": ["gini", "entropy"],
    "max_depth": [2, 3, 4, 5, 6, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5, 10],
    "ccp_alpha": [0.0, 0.001, 0.01]
}

inicio = time.perf_counter()

grid_arbol = GridSearchCV(
    estimator=arbol,
    param_grid=param_grid_arbol,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid_arbol.fit(X_train, y_train)

tiempo_arbol = time.perf_counter() - inicio
mejor_arbol = grid_arbol.best_estimator_

print("Mejores hiperparámetros Árbol:")
print(grid_arbol.best_params_)

print("\nMejor F1 macro promedio CV:")
print(round(grid_arbol.best_score_, 4))

print("\nTiempo GridSearch:")
print(round(tiempo_arbol, 3), "segundos")


# 8. Optimización de Random Forest

In [ ]:

rf = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1
)

param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [4, None],
    "min_samples_leaf": [1, 4],
    "max_features": ["sqrt", 0.7],
    "criterion": ["gini", "entropy"]
}

inicio = time.perf_counter()

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

grid_rf.fit(X_train, y_train)

tiempo_rf = time.perf_counter() - inicio
mejor_rf = grid_rf.best_estimator_

print("Mejores hiperparámetros Random Forest:")
print(grid_rf.best_params_)

print("\nMejor F1 macro promedio CV:")
print(round(grid_rf.best_score_, 4))

print("\nTiempo GridSearch:")
print(round(tiempo_rf, 3), "segundos")



# 9. Evaluación final en Train y Test

Después de seleccionar los mejores hiperparámetros usando Train + CV, ambos modelos se evalúan sobre el mismo Test.


In [ ]:

# Árbol
pred_arbol_train = mejor_arbol.predict(X_train)
pred_arbol_test = mejor_arbol.predict(X_test)

proba_arbol_train = mejor_arbol.predict_proba(X_train)[:, 1]
proba_arbol_test = mejor_arbol.predict_proba(X_test)[:, 1]

# Random Forest
pred_rf_train = mejor_rf.predict(X_train)
pred_rf_test = mejor_rf.predict(X_test)

proba_rf_train = mejor_rf.predict_proba(X_train)[:, 1]
proba_rf_test = mejor_rf.predict_proba(X_test)[:, 1]

# Métricas
met_arbol_train = metricas_clasificacion(
    y_train, pred_arbol_train, proba_arbol_train
)
met_arbol_test = metricas_clasificacion(
    y_test, pred_arbol_test, proba_arbol_test
)

met_rf_train = metricas_clasificacion(
    y_train, pred_rf_train, proba_rf_train
)
met_rf_test = metricas_clasificacion(
    y_test, pred_rf_test, proba_rf_test
)

tabla_test = pd.DataFrame({
    "Árbol de Decisión": met_arbol_test,
    "Random Forest": met_rf_test
}).T

display(tabla_test.round(4))


# 10. ¿Quién gana en cada métrica?

In [ ]:

ganadores = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Precision macro",
        "Recall macro",
        "F1 macro",
        "ROC-AUC",
        "Recall malignant"
    ],
    "Mejor modelo": [
        tabla_test["Accuracy"].idxmax(),
        tabla_test["Precision_macro"].idxmax(),
        tabla_test["Recall_macro"].idxmax(),
        tabla_test["F1_macro"].idxmax(),
        tabla_test["ROC_AUC"].idxmax(),
        tabla_test["Recall_malignant"].idxmax()
    ]
})

display(ganadores)


In [ ]:

# Comparación de métricas principales
metricas_principales = [
    "Accuracy",
    "Precision_macro",
    "Recall_macro",
    "F1_macro",
    "ROC_AUC",
    "Recall_malignant"
]

x = np.arange(len(metricas_principales))
ancho = 0.35

plt.figure(figsize=(12,6))

plt.bar(
    x - ancho/2,
    tabla_test.loc["Árbol de Decisión", metricas_principales],
    width=ancho,
    label="Árbol de Decisión"
)

plt.bar(
    x + ancho/2,
    tabla_test.loc["Random Forest", metricas_principales],
    width=ancho,
    label="Random Forest"
)

plt.xticks(
    x,
    [
        "Accuracy",
        "Precision\nmacro",
        "Recall\nmacro",
        "F1\nmacro",
        "ROC-AUC",
        "Recall\nmalignant"
    ]
)

plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Comparación de métricas en Test")
plt.legend()
plt.show()


# 11. Train vs Test: ¿quién sobreajusta más?

In [ ]:

comparacion_train_test = pd.DataFrame({
    "Modelo": [
        "Árbol de Decisión",
        "Random Forest"
    ],
    "F1 Train": [
        met_arbol_train["F1_macro"],
        met_rf_train["F1_macro"]
    ],
    "F1 Test": [
        met_arbol_test["F1_macro"],
        met_rf_test["F1_macro"]
    ],
    "Accuracy Train": [
        met_arbol_train["Accuracy"],
        met_rf_train["Accuracy"]
    ],
    "Accuracy Test": [
        met_arbol_test["Accuracy"],
        met_rf_test["Accuracy"]
    ]
})

display(comparacion_train_test.round(4))


In [ ]:

x = np.arange(2)
ancho = 0.35

plt.figure(figsize=(8,5))
plt.bar(
    x-ancho/2,
    comparacion_train_test["F1 Train"],
    width=ancho,
    label="Train"
)
plt.bar(
    x+ancho/2,
    comparacion_train_test["F1 Test"],
    width=ancho,
    label="Test"
)
plt.xticks(x, comparacion_train_test["Modelo"])
plt.ylim(0,1.05)
plt.ylabel("F1 macro")
plt.title("F1 macro Train vs Test")
plt.legend()
plt.show()



Una brecha grande entre Train y Test puede sugerir overfitting.

No basta con que un modelo tenga Train perfecto: debe generalizar bien.


# 12. Validación cruzada de los modelos optimizados

In [ ]:

scoring = {
    "Accuracy": "accuracy",
    "F1_macro": "f1_macro",
    "Recall_macro": "recall_macro",
    "Precision_macro": "precision_macro",
    "ROC_AUC": "roc_auc"
}

cv_arbol = cross_validate(
    mejor_arbol,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

cv_rf = cross_validate(
    mejor_rf,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

tabla_cv = pd.DataFrame({
    "Árbol_F1": cv_arbol["test_F1_macro"],
    "RF_F1": cv_rf["test_F1_macro"],
    "Árbol_Accuracy": cv_arbol["test_Accuracy"],
    "RF_Accuracy": cv_rf["test_Accuracy"],
    "Árbol_AUC": cv_arbol["test_ROC_AUC"],
    "RF_AUC": cv_rf["test_ROC_AUC"]
})

display(tabla_cv.round(4))


In [ ]:

resumen_cv = pd.DataFrame({
    "Árbol de Decisión": {
        "CV_F1_macro_promedio": cv_arbol["test_F1_macro"].mean(),
        "CV_Accuracy_promedio": cv_arbol["test_Accuracy"].mean(),
        "CV_ROC_AUC_promedio": cv_arbol["test_ROC_AUC"].mean(),
        "CV_F1_sd": cv_arbol["test_F1_macro"].std(ddof=1)
    },
    "Random Forest": {
        "CV_F1_macro_promedio": cv_rf["test_F1_macro"].mean(),
        "CV_Accuracy_promedio": cv_rf["test_Accuracy"].mean(),
        "CV_ROC_AUC_promedio": cv_rf["test_ROC_AUC"].mean(),
        "CV_F1_sd": cv_rf["test_F1_macro"].std(ddof=1)
    }
}).T

display(resumen_cv.round(4))


In [ ]:

folds = np.arange(1, 6)

plt.figure(figsize=(9,5))
plt.plot(
    folds,
    cv_arbol["test_F1_macro"],
    marker="o",
    label="Árbol de Decisión"
)
plt.plot(
    folds,
    cv_rf["test_F1_macro"],
    marker="o",
    label="Random Forest"
)

plt.xticks(folds)
plt.xlabel("Fold")
plt.ylabel("F1 macro")
plt.title("F1 macro por fold")
plt.ylim(0,1.05)
plt.legend()
plt.grid(alpha=0.2)
plt.show()


# 13. Matriz de confusión — Árbol de Decisión

In [ ]:

cm_arbol = confusion_matrix(
    y_test,
    pred_arbol_test
)

ConfusionMatrixDisplay(
    confusion_matrix=cm_arbol,
    display_labels=data.target_names
).plot()

plt.title("Matriz de confusión — Árbol de Decisión")
plt.show()


# 14. Matriz de confusión — Random Forest

In [ ]:

cm_rf = confusion_matrix(
    y_test,
    pred_rf_test
)

ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=data.target_names
).plot()

plt.title("Matriz de confusión — Random Forest")
plt.show()



# 15. Falsos positivos y falsos negativos

Como las clases son:

- 0 = malignant
- 1 = benign

es importante revisar directamente los errores de cada modelo.


In [ ]:

tn_a, fp_a, fn_a, tp_a = cm_arbol.ravel()
tn_r, fp_r, fn_r, tp_r = cm_rf.ravel()

errores_confusion = pd.DataFrame({
    "Árbol de Decisión": {
        "TN": tn_a,
        "FP": fp_a,
        "FN": fn_a,
        "TP": tp_a
    },
    "Random Forest": {
        "TN": tn_r,
        "FP": fp_r,
        "FN": fn_r,
        "TP": tp_r
    }
}).T

display(errores_confusion)



> Debido a la codificación del dataset, interpretar TP/FN exige recordar que la clase `1` es benign.  
> Para la detección de malignidad, el indicador más directo que mostramos es **Recall_malignant**.


# 16. Curvas ROC

In [ ]:

fpr_arbol, tpr_arbol, _ = roc_curve(
    y_test,
    proba_arbol_test
)

fpr_rf, tpr_rf, _ = roc_curve(
    y_test,
    proba_rf_test
)

auc_arbol = roc_auc_score(
    y_test,
    proba_arbol_test
)

auc_rf = roc_auc_score(
    y_test,
    proba_rf_test
)

plt.figure(figsize=(8,6))
plt.plot(
    fpr_arbol,
    tpr_arbol,
    label=f"Árbol (AUC={auc_arbol:.3f})"
)
plt.plot(
    fpr_rf,
    tpr_rf,
    label=f"Random Forest (AUC={auc_rf:.3f})"
)
plt.plot(
    [0,1],
    [0,1],
    linestyle="--",
    label="Azar"
)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Comparación de curvas ROC")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


# 17. Curvas Precision-Recall

In [ ]:

precision_a, recall_a, _ = precision_recall_curve(
    y_test,
    proba_arbol_test
)

precision_r, recall_r, _ = precision_recall_curve(
    y_test,
    proba_rf_test
)

ap_arbol = average_precision_score(
    y_test,
    proba_arbol_test
)

ap_rf = average_precision_score(
    y_test,
    proba_rf_test
)

plt.figure(figsize=(8,6))
plt.plot(
    recall_a,
    precision_a,
    label=f"Árbol (AP={ap_arbol:.3f})"
)
plt.plot(
    recall_r,
    precision_r,
    label=f"Random Forest (AP={ap_rf:.3f})"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Comparación Precision-Recall")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


# 18. Comparación observación por observación

In [ ]:

comparacion_predicciones = pd.DataFrame({
    "Real": y_test.to_numpy(),
    "Árbol": pred_arbol_test,
    "Random Forest": pred_rf_test,
    "Prob_Árbol_clase1": proba_arbol_test,
    "Prob_RF_clase1": proba_rf_test
})

comparacion_predicciones["Árbol_correcto"] = (
    comparacion_predicciones["Real"]
    == comparacion_predicciones["Árbol"]
)

comparacion_predicciones["RF_correcto"] = (
    comparacion_predicciones["Real"]
    == comparacion_predicciones["Random Forest"]
)

comparacion_predicciones["Resultado_comparativo"] = np.select(
    [
        comparacion_predicciones["Árbol_correcto"]
        & comparacion_predicciones["RF_correcto"],

        comparacion_predicciones["Árbol_correcto"]
        & ~comparacion_predicciones["RF_correcto"],

        ~comparacion_predicciones["Árbol_correcto"]
        & comparacion_predicciones["RF_correcto"]
    ],
    [
        "Ambos correctos",
        "Solo Árbol correcto",
        "Solo Random Forest correcto"
    ],
    default="Ambos incorrectos"
)

display(comparacion_predicciones.head(15))


In [ ]:

conteo_comparativo = (
    comparacion_predicciones["Resultado_comparativo"]
    .value_counts()
)

display(
    conteo_comparativo
    .to_frame("Número de observaciones")
)

plt.figure(figsize=(9,5))
plt.bar(
    conteo_comparativo.index,
    conteo_comparativo.values
)
plt.title("Comparación predicción por predicción")
plt.ylabel("Observaciones de Test")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


# 19. Predicción individual con ambos modelos

In [ ]:

posicion = 0

observacion = X_test.iloc[[posicion]]
valor_real = y_test.iloc[posicion]

pred_a = mejor_arbol.predict(observacion)[0]
pred_r = mejor_rf.predict(observacion)[0]

prob_a = mejor_arbol.predict_proba(observacion)[0]
prob_r = mejor_rf.predict_proba(observacion)[0]

print("CLASE REAL:")
print(data.target_names[valor_real])

print("\nÁRBOL DE DECISIÓN")
print("Predicción:", data.target_names[pred_a])
for clase, prob in zip(data.target_names, prob_a):
    print(f"{clase}: {prob:.2%}")

print("\nRANDOM FOREST")
print("Predicción:", data.target_names[pred_r])
for clase, prob in zip(data.target_names, prob_r):
    print(f"{clase}: {prob:.2%}")


# 20. Importancia de variables

In [ ]:

imp_arbol = pd.Series(
    mejor_arbol.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

imp_rf = pd.Series(
    mejor_rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

tabla_importancias = pd.DataFrame({
    "Árbol": imp_arbol,
    "Random Forest": imp_rf
}).fillna(0)

display(
    tabla_importancias
    .sort_values("Random Forest", ascending=False)
    .head(15)
)


In [ ]:

top_rf = imp_rf.head(15)

plt.figure(figsize=(10,6))
plt.barh(
    top_rf.index[::-1],
    top_rf.values[::-1]
)
plt.title("15 variables más importantes — Random Forest")
plt.xlabel("Importancia")
plt.tight_layout()
plt.show()



> La importancia de variables es predictiva. No demuestra causalidad.


# 21. Coste computacional

In [ ]:

coste = pd.DataFrame({
    "Modelo": [
        "Árbol de Decisión",
        "Random Forest"
    ],
    "Tiempo GridSearch (s)": [
        tiempo_arbol,
        tiempo_rf
    ],
    "Número de árboles": [
        1,
        mejor_rf.n_estimators
    ]
})

display(coste.round(3))



# 22. Selección automática del mejor modelo

Usaremos como criterio principal:

1. mayor **F1 macro en Test**;
2. revisaremos Accuracy, ROC-AUC y Recall malignant;
3. verificaremos que la conclusión sea consistente con validación cruzada.


In [ ]:

ganador_f1 = (
    "Árbol de Decisión"
    if met_arbol_test["F1_macro"] > met_rf_test["F1_macro"]
    else "Random Forest"
)

f1_mejor = max(
    met_arbol_test["F1_macro"],
    met_rf_test["F1_macro"]
)

f1_otro = min(
    met_arbol_test["F1_macro"],
    met_rf_test["F1_macro"]
)

mejora_f1 = (
    (f1_mejor - f1_otro)
    / f1_otro
    * 100
)

print("MEJOR MODELO SEGÚN F1 MACRO EN TEST:")
print(ganador_f1)

print("\nMejora relativa de F1 macro:")
print(round(mejora_f1, 2), "%")

print("\n--- ÁRBOL DE DECISIÓN ---")
for k, v in met_arbol_test.items():
    print(k, ":", round(v, 4))

print("\n--- RANDOM FOREST ---")
for k, v in met_rf_test.items():
    print(k, ":", round(v, 4))



# 23. ¿Cómo interpretar el resultado?

## Si Random Forest obtiene:

- mayor F1 macro;
- mayor ROC-AUC;
- mayor Accuracy;
- mejor Recall malignant;
- mejor consistencia entre folds;

entonces existe evidencia de que **generaliza mejor en este ejercicio**.

## Si el Árbol obtiene resultados cercanos

Puede seguir siendo útil cuando la interpretabilidad sea prioritaria.

La selección final depende de:

- capacidad predictiva;
- costo de falsos negativos;
- costo de falsos positivos;
- interpretabilidad;
- costo computacional.



# 24. Resumen conceptual

| Criterio | Árbol de Decisión | Random Forest |
|---|---|---|
| Árboles | 1 | Muchos |
| Interpretabilidad | Muy alta | Menor |
| Estabilidad | Menor | Mayor |
| Varianza | Mayor | Menor |
| Overfitting | Más probable | Generalmente menor |
| Probabilidades | Sí | Sí |
| Feature importance | Sí | Sí |
| OOB Score | No | Sí |
| Coste computacional | Bajo | Mayor |



# 25. Conclusión

La comparación correcta utiliza:

**mismos datos + mismo Train/Test + misma CV + mismas métricas**

y después pregunta:

> **¿Cuál clasifica mejor sin sacrificar de forma importante la detección de la clase que más nos interesa?**

En este notebook la métrica principal es F1 macro, pero también analizamos ROC-AUC, Accuracy y Recall de malignidad para obtener una conclusión más completa.



# 26. Referencias

- Scikit-learn — DecisionTreeClassifier  
  https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

- Scikit-learn — RandomForestClassifier  
  https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

- Scikit-learn — Breast Cancer Wisconsin  
  https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html

- Scikit-learn — Classification metrics  
  https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics
